 # 03 — Feature Engineering
 Inputs:
 - data/processed/fragrantica_clean.csv
 
 Outputs:
 - data/processed/fragrantica_features.parquet
 - data/processed/fragrantica_features.csv

 Features:
 - Targets: weighted_rating
 - Popularity: log_rating_count, popularity_class
 - Notes: parsed lists + top-N note one-hot
 - Accords: top-K accord one-hot - Metadata: perfume_age, note counts
 - Brand aggregates: count + mean stats (note: leakage-safe version comes in modeling notebook)

In [1]:
# Imports

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# Setting paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "fragrantica_clean.csv"
OUT_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PARQUET = OUT_DIR / "fragrantica_features.parquet"
FEATURES_CSV = OUT_DIR / "fragrantica_features.csv"

CLEAN_PATH, FEATURES_PARQUET

(WindowsPath('c:/Users/swara/olfactory-intelligence/data/processed/fragrantica_clean.csv'),
 WindowsPath('c:/Users/swara/olfactory-intelligence/data/processed/fragrantica_features.parquet'))

In [3]:
# Load cleaned data

df = pd.read_csv(CLEAN_PATH)
df.shape, df.head(3)

((23846, 18),
                                                  url              perfume  \
 0  https://www.fragrantica.com/perfume/a-dozen-ro...          amber-queen   
 1  https://www.fragrantica.com/perfume/a-dozen-ro...           angel-face   
 2  https://www.fragrantica.com/perfume/a-dozen-ro...  shakespeare-in-love   
 
            brand country gender  rating_value  rating_count    year  \
 0  a-dozen-roses     USA  women          3.77            96  2012.0   
 1  a-dozen-roses     USA  women          3.96           107  2013.0   
 2  a-dozen-roses     USA  women          3.62           109  2011.0   
 
                            top                               middle  \
 0  apricot, clementine, ginger                    damask rose, rose   
 1         black currant, apple  rose, violet, peony, lilac, jasmine   
 2                         pear              rose, gardenia, jasmine   
 
                              base    perfumer1 perfumer2 mainaccord1  \
 0                 

In [4]:
# Safety checks

required = [
    "url","perfume","brand","country","gender","rating_value","rating_count","year",
    "top","middle","base",
    "mainaccord1","mainaccord2","mainaccord3","mainaccord4","mainaccord5",
    "perfumer1","perfumer2"
]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"

assert df["rating_value"].notna().all()
assert df["rating_count"].notna().all()
assert (df["rating_count"] > 0).all()

In [5]:
#Helper functions

#Convert note strings -> lists
def safe_note_list(x) -> list[str]:
    if pd.isna(x):
        return []
    s = str(x).strip().lower()
    if not s:
        return []
    parts = [p.strip() for p in s.split(",")]
    return [p for p in parts if p]

# Combine top, middle and base notes
def combine_unique(*lists: list[list[str]]) -> list[str]:
    seen = set()
    out = []
    for lst in lists:
        for item in lst:
            if item not in seen:
                seen.add(item)
                out.append(item)
    return out

# Computing weighted ratings
def imdb_weighted_rating(R: pd.Series, v: pd.Series, m: float, C: float) -> pd.Series:
    v = v.astype(float)
    R = R.astype(float)
    return (v / (v + m)) * R + (m / (v + m)) * C

In [6]:
# Create target variables

# Weighted rating
C = float(df["rating_value"].mean())
m = float(df["rating_count"].quantile(0.75))

df["weighted_rating"] = imdb_weighted_rating(
    R=df["rating_value"],
    v=df["rating_count"],
    m=m,
    C=C,
)
# Log rating_count 
df["log_rating_count"] = np.log1p(df["rating_count"].astype(float))

C, m, df[["rating_value","rating_count","weighted_rating","log_rating_count"]].head(5)

(3.9605258743604796,
 362.0,
    rating_value  rating_count  weighted_rating  log_rating_count
 0          3.77            96         3.920590          4.574711
 1          3.96           107         3.960406          4.682131
 2          3.62           109         3.881721          4.700480
 3          3.68           148         3.879118          5.003946
 4          4.25            91         4.018676          4.521789)

In [7]:
# Create popularity tiers based on rating_count

q1, q2 = df["rating_count"].quantile([0.33, 0.66]).values

def popularity_class(v: int) -> str:
    if v <= q1:
        return "low"
    if v <= q2:
        return "mid"
    return "high"

df["popularity_class"] = df["rating_count"].apply(popularity_class).astype("category")
df["popularity_class"].value_counts()


popularity_class
high    8077
low     7935
mid     7834
Name: count, dtype: int64

In [8]:
# Metadata / Age Feature Engineering

CURRENT_YEAR = 2026


df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Create missing year flag
df["year_missing"] = df["year"].isna().astype("Int64")

# Compute perfume_age 
df["perfume_age"] = CURRENT_YEAR - df["year"]

# Remove unrealistic ages (negative or extreme future values just in case)
df.loc[df["perfume_age"] < 0, "perfume_age"] = pd.NA

# Convert to nullable float (so NaN is preserved properly)
df["perfume_age"] = df["perfume_age"].astype("Float64")

# Additional simple metadata features
df["brand_len"] = df["brand"].astype(str).str.len().astype("Int64")
df["perfume_name_len"] = df["perfume"].astype(str).str.len().astype("Int64")

# Quick inspection
df[["year", "year_missing", "perfume_age", "brand_len", "perfume_name_len"]].head(10)

,year,year_missing,perfume_age,brand_len,perfume_name_len
0,2012.0,0,14.0,13,11
1,2013.0,0,13.0,13,10
2,2011.0,0,15.0,13,19
3,2018.0,0,8.0,13,20
4,2018.0,0,8.0,13,8
5,2012.0,0,14.0,13,11
6,2017.0,0,9.0,13,24
7,2013.0,0,13.0,13,20
8,2018.0,0,8.0,9,5
9,2018.0,0,8.0,9,5


In [9]:
# Parse notes + complexity features

# Parse raw note strings into lists
df["top_notes"] = df["top"].apply(safe_note_list)
df["middle_notes"] = df["middle"].apply(safe_note_list)
df["base_notes"] = df["base"].apply(safe_note_list)

# Combine all notes into one list
df["all_notes"] = df.apply(
    lambda r: combine_unique(r["top_notes"], r["middle_notes"], r["base_notes"]),
    axis=1
)

#  Count the notes
df["top_note_count"] = df["top_notes"].apply(len).astype("Int64")
df["middle_note_count"] = df["middle_notes"].apply(len).astype("Int64")
df["base_note_count"] = df["base_notes"].apply(len).astype("Int64")
df["note_count_total"] = df["all_notes"].apply(len).astype("Int64")

df[["top_note_count","middle_note_count","base_note_count","note_count_total"]].describe()

,top_note_count,middle_note_count,base_note_count,note_count_total
count,23846.0,23846.0,23846.0,23846.0
mean,3.110123,3.299589,3.497903,9.850289
std,1.495822,1.614385,1.845732,4.129836
min,1.0,1.0,1.0,1.0
25%,2.0,2.0,2.0,7.0
50%,3.0,3.0,3.0,9.0
75%,4.0,4.0,4.0,12.0
max,27.0,25.0,22.0,62.0


In [10]:
# Additional structure-based note features

# Avoid division by zero
note_total_safe = df["note_count_total"].replace(0, np.nan)

df["top_to_total_ratio"] = (df["top_note_count"] / note_total_safe).fillna(0)
df["mid_to_total_ratio"] = (df["middle_note_count"] / note_total_safe).fillna(0)
df["base_to_total_ratio"] = (df["base_note_count"] / note_total_safe).fillna(0)

df[["top_to_total_ratio", "mid_to_total_ratio", "base_to_total_ratio"]].head()

,top_to_total_ratio,mid_to_total_ratio,base_to_total_ratio
0,0.5,0.333333,0.166667
1,0.2,0.5,0.3
2,0.166667,0.5,0.333333
3,0.375,0.25,0.375
4,0.285714,0.285714,0.428571


In [11]:
# Notes one-hot encoding (top-N notes only)

# Count note frequencies
note_freq = df["all_notes"].explode().value_counts()

# Only keep top 300 notes
TOP_N_NOTES = 300
top_notes_vocab = set(note_freq.head(TOP_N_NOTES).index)

# Filter each perfumes notes
df["all_notes_topN"] = df["all_notes"].apply(lambda lst: [n for n in lst if n in top_notes_vocab])

# One hot encode
mlb_notes = MultiLabelBinarizer()
note_ohe = mlb_notes.fit_transform(df["all_notes_topN"])

# Convert into DataFrame
note_ohe_df = pd.DataFrame(
    note_ohe,
    columns=[f"note__{c}" for c in mlb_notes.classes_],
    index=df.index
)

note_ohe_df.shape

(23846, 300)

In [13]:
# Power note features

power_notes = [
    "note__vanilla",
    "note__oud",
    "note__rose",
    "note__musk",
    "note__sandalwood"
]

existing_power_notes = [c for c in power_notes if c in note_ohe_df.columns]

if existing_power_notes:
    df["has_power_note"] = note_ohe_df[existing_power_notes].any(axis=1).astype("Int64")
    df["power_note_count"] = note_ohe_df[existing_power_notes].sum(axis=1).astype("Int64")
else:
    df["has_power_note"] = 0
    df["power_note_count"] = 0

df[["has_power_note", "power_note_count"]].head()

,has_power_note,power_note_count
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1


In [14]:
# Main accords one-hot encoding

# Clean accord columns
accord_cols = ["mainaccord1","mainaccord2","mainaccord3","mainaccord4","mainaccord5"]

for c in accord_cols:
    df[c] = df[c].astype("string").str.strip().str.lower()

#Combine accords into one list per perfume
df["all_accords"] = df[accord_cols].apply(
    lambda r: [x for x in r.tolist() if pd.notna(x) and str(x).strip() != ""],
    axis=1
)

# Count frequencies
acc_freq = df["all_accords"].explode().value_counts()

# Only keep top 30
TOP_K_ACCORDS = 30
top_acc_vocab = set(acc_freq.head(TOP_K_ACCORDS).index)

df["all_accords_topK"] = df["all_accords"].apply(lambda lst: [a for a in lst if a in top_acc_vocab])

# Binary matrix
mlb_acc = MultiLabelBinarizer()
acc_ohe = mlb_acc.fit_transform(df["all_accords_topK"])

# Wrap into DataFrame
acc_ohe_df = pd.DataFrame(
    acc_ohe,
    columns=[f"accord__{c}" for c in mlb_acc.classes_],
    index=df.index
)

acc_ohe_df.shape

(23846, 30)

In [15]:
# Accord complexity feature

df["accord_count"] = df["all_accords_topK"].apply(len).astype("Int64")

df[["accord_count"]].describe()

,accord_count
count,23846.0
mean,4.604294
std,0.662639
min,1.0
25%,4.0
50%,5.0
75%,5.0
max,5.0


In [16]:
# Encode gender and country (with country grouping)

gender_ohe_df = pd.get_dummies(df["gender"], prefix="gender", dummy_na=True)

country_counts = df["country"].astype(str).str.strip().value_counts()
TOP_COUNTRIES = 25
top_countries = set(country_counts.head(TOP_COUNTRIES).index)

df["country_top"] = df["country"].where(df["country"].isin(top_countries), other="other")
country_ohe_df = pd.get_dummies(df["country_top"], prefix="country")

gender_ohe_df.shape, country_ohe_df.shape

((23846, 4), (23846, 25))

In [17]:
# Brand aggregated features

brand_group = df.groupby("brand", dropna=False)

df["brand_perfume_count"] = brand_group["url"].transform("count").astype("Int64")
df["brand_mean_rating"] = brand_group["rating_value"].transform("mean").astype(float)
df["brand_mean_weighted_rating"] = brand_group["weighted_rating"].transform("mean").astype(float)
df["brand_mean_log_votes"] = brand_group["log_rating_count"].transform("mean").astype(float)

df[["brand","brand_perfume_count","brand_mean_rating"]].head(5)


,brand,brand_perfume_count,brand_mean_rating
0,a-dozen-roses,3,3.783333
1,a-dozen-roses,3,3.783333
2,a-dozen-roses,3,3.783333
3,a-lab-on-fire,5,3.940000
4,a-lab-on-fire,5,3.940000


In [19]:
# Assemble final features table

base_cols = [
    "url","perfume","brand","country","country_top","gender","year",
    "rating_value","rating_count","weighted_rating","log_rating_count",
    "perfume_age","brand_len","perfume_name_len",
    "top_note_count","middle_note_count","base_note_count","note_count_total",
    "top_to_total_ratio","mid_to_total_ratio","base_to_total_ratio",
    "accord_count",
    "has_power_note","power_note_count",
    "brand_perfume_count","brand_mean_rating","brand_mean_weighted_rating","brand_mean_log_votes",
    "popularity_class",
]
features = pd.concat(
    [
        df[base_cols],
        gender_ohe_df,
        country_ohe_df,
        acc_ohe_df,
        note_ohe_df,
    ],
    axis=1
)

features.shape

(23846, 388)

In [20]:
# Check non-numeric columns

non_numeric = features.select_dtypes(exclude=["number","bool"]).columns.tolist()
non_numeric


['url',
 'perfume',
 'brand',
 'country',
 'country_top',
 'gender',
 'popularity_class']

In [22]:
# Create model-ready X and y

id_cols = ["url","perfume","brand","country","country_top","gender","popularity_class"]
target_cols = ["weighted_rating","rating_value","log_rating_count"]

X = features.drop(columns=id_cols + target_cols, errors="ignore")
y = features["weighted_rating"].astype(float)

X.shape, y.shape

((23846, 378), (23846,))

In [23]:
# Save outputs
features.to_parquet(FEATURES_PARQUET, index=False)
features.to_csv(FEATURES_CSV, index=False, encoding="utf-8")

FEATURES_PARQUET, FEATURES_CSV

(WindowsPath('c:/Users/swara/olfactory-intelligence/data/processed/fragrantica_features.parquet'),
 WindowsPath('c:/Users/swara/olfactory-intelligence/data/processed/fragrantica_features.csv'))

(300,
 ['note__african orange flower',
  'note__agarwood (oud)',
  'note__aldehydes',
  'note__almond',
  'note__almond blossom',
  'note__amalfi lemon',
  'note__amber',
  'note__ambergris',
  'note__amberwood',
  'note__ambrette (musk mallow)'])